In [7]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

from __future__ import annotations

import argparse
import importlib
import json
import sys
import time
from pathlib import Path
from typing import Any

import undetected_chromedriver as uc


# ==================================================
# Notebookテスト用店舗
# ==================================================
#
# Notebookで実行する場合は、ここだけ変更する。
#
# .pyで実行する場合は、
# --siteで指定した店舗が優先される。
#
# 例:
# python cookie_save.py --site iwakuni_tekisasu_s
#

NOTEBOOK_SITE = "iwakuni_tekisasu_s"


# ==================================================
# ブラウザ設定
# ==================================================

PAGE_WAIT_SECONDS = 30

WINDOW_WIDTH = 600
WINDOW_HEIGHT = 1000

# NoneならChromeバージョンを自動判定
CHROME_VERSION_MAIN: int | None = 139


# ==================================================
# プロジェクトルート検出
# ==================================================

def find_project_root(
    start_path: Path,
) -> Path:
    """
    config/ と utils/ がある場所を
    プロジェクトルートとして返す。
    """
    current = start_path.resolve()

    if current.is_file():
        current = current.parent

    for candidate in (
        current,
        *current.parents,
    ):
        if (
            (candidate / "config").is_dir()
            and (candidate / "utils").is_dir()
        ):
            return candidate

    raise RuntimeError(
        "PROJECT_ROOTを特定できませんでした。"
        f" 開始位置: {start_path}"
    )


if "__file__" in globals():
    PROJECT_ROOT = find_project_root(
        Path(__file__)
    )
else:
    PROJECT_ROOT = find_project_root(
        Path.cwd()
    )


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


print(f"[INFO] PROJECT_ROOT: {PROJECT_ROOT}")
print(
    f"[INFO] config存在: "
    f"{(PROJECT_ROOT / 'config').is_dir()}"
)
print(
    f"[INFO] utils存在: "
    f"{(PROJECT_ROOT / 'utils').is_dir()}"
)


# ==================================================
# 共通設定
# ==================================================

from config.common import DEFAULT_SITE


# ==================================================
# 店舗選択
# ==================================================

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description=(
            "店舗設定のCOOKIE_URLへアクセスし、"
            "Cookieを保存します。"
        )
    )

    parser.add_argument(
        "--site",
        default=DEFAULT_SITE,
        help="configフォルダ内の店舗設定名",
    )

    return parser.parse_args()


if "__file__" in globals():
    # .py実行時は--siteを使用
    args = parse_args()
else:
    # Notebook実行時は上部のNOTEBOOK_SITEを使用
    args = argparse.Namespace(
        site=NOTEBOOK_SITE,
    )


site_name = str(
    args.site
).strip()


if not site_name:
    raise ValueError(
        "店舗設定名が空です。"
    )


# ==================================================
# 店舗設定読み込み
# ==================================================

config_file = (
    PROJECT_ROOT
    / "config"
    / f"{site_name}.py"
)


if not config_file.is_file():
    raise FileNotFoundError(
        f"店舗設定が見つかりません: "
        f"{config_file}"
    )


try:
    site_config = importlib.import_module(
        f"config.{site_name}"
    )

except ModuleNotFoundError as exc:
    raise SystemExit(
        "[ERROR] 店舗設定の読み込みに失敗しました: "
        f"config/{site_name}.py"
    ) from exc


# ==================================================
# 必須設定確認
# ==================================================

required_settings = (
    "COOKIE_URL",
    "COOKIE_FILE",
)


for setting_name in required_settings:
    if not hasattr(
        site_config,
        setting_name,
    ):
        raise AttributeError(
            f"config/{site_name}.py に "
            f"{setting_name} が設定されていません。"
        )


# ==================================================
# 店舗別設定
# ==================================================

shop_name = str(
    getattr(
        site_config,
        "SHOP_NAME",
        getattr(
            site_config,
            "GSHEET_NAME",
            site_name,
        ),
    )
).strip()


cookie_url = str(
    site_config.COOKIE_URL
).strip()


cookie_file = Path(
    site_config.COOKIE_FILE
)


if not cookie_url.startswith(
    (
        "http://",
        "https://",
    )
):
    raise ValueError(
        f"COOKIE_URLが正しいURLではありません: "
        f"{cookie_url}"
    )


cookie_file.parent.mkdir(
    parents=True,
    exist_ok=True,
)


print(f"[INFO] 対象店舗設定: {site_name}")
print(f"[INFO] 店舗名: {shop_name}")
print(f"[INFO] Cookie取得URL: {cookie_url}")
print(f"[INFO] Cookie保存先: {cookie_file}")


# ==================================================
# Cookie保存
# ==================================================

def save_cookies(
    browser: Any,
    destination: Path,
) -> None:
    """
    現在のブラウザのCookieをJSONへ保存する。
    """
    cookies = browser.get_cookies()

    with destination.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            cookies,
            file,
            ensure_ascii=False,
            indent=2,
        )

    print(
        f"[COOKIE] 取得件数: "
        f"{len(cookies)}件"
    )

    print(
        f"✅ Cookie保存完了: "
        f"{destination}"
    )


# ==================================================
# ブラウザ起動
# ==================================================

def open_browser() -> Any:
    """
    undetected_chromedriverでChromeを起動する。
    """
    chrome_options = uc.ChromeOptions()

    print("[BROWSER] Chrome起動開始")

    chrome_arguments: dict[str, Any] = {
        "options": chrome_options,
    }

    if CHROME_VERSION_MAIN is not None:
        chrome_arguments[
            "version_main"
        ] = CHROME_VERSION_MAIN

        print(
            "[BROWSER] Chromeバージョン指定: "
            f"{CHROME_VERSION_MAIN}"
        )

    else:
        print(
            "[BROWSER] Chromeバージョン: "
            "自動判定"
        )

    browser = uc.Chrome(
        **chrome_arguments
    )

    browser.set_window_size(
        WINDOW_WIDTH,
        WINDOW_HEIGHT,
    )

    print("[BROWSER] Chrome起動完了")

    return browser


# ==================================================
# メイン処理
# ==================================================

def main() -> None:
    start_time = time.time()
    browser = None

    try:
        browser = open_browser()

        print(
            f"[NAV] Cookie取得URLへアクセス: "
            f"{cookie_url}"
        )

        browser.get(
            cookie_url
        )

        print(
            f"[WAIT] ページ描画待機: "
            f"{PAGE_WAIT_SECONDS}秒"
        )

        time.sleep(
            PAGE_WAIT_SECONDS
        )

        print(
            f"[NAV] 現在URL: "
            f"{browser.current_url}"
        )

        print(
            f"[NAV] ページタイトル: "
            f"{browser.title}"
        )

        print("[COOKIE] Cookie取得開始")

        save_cookies(
            browser,
            cookie_file,
        )

    finally:
        if browser is not None:
            print(
                "[BROWSER] browser.quit() 開始"
            )

            try:
                browser.quit()

                print(
                    "[BROWSER] browser.quit() 完了"
                )

            except Exception as quit_error:
                print(
                    "[WARN] ブラウザ終了失敗: "
                    f"{type(quit_error).__name__}: "
                    f"{quit_error}"
                )

        elapsed_time = (
            time.time()
            - start_time
        )

        print(
            f"[TIME] 実行時間: "
            f"{elapsed_time:.2f}秒"
        )


# ==================================================
# 実行
# ==================================================

if __name__ == "__main__":
    main()

[INFO] PROJECT_ROOT: /home/ubuntu/myenv310/detaslot
[INFO] config存在: True
[INFO] utils存在: True
[INFO] 対象店舗設定: iwakuni_tekisasu_s
[INFO] 店舗名: iwakuni_tekisasu_s
[INFO] Cookie取得URL: https://www.pscube.jp/dedamajyoho-P-townDMMpachi/c734011
[INFO] Cookie保存先: /home/ubuntu/myenv310/detaslot/credentials/iwakuni_tekisasu_s/cookies.json
[BROWSER] Chrome起動開始
[BROWSER] Chromeバージョン指定: 139
[BROWSER] Chrome起動完了
[NAV] Cookie取得URLへアクセス: https://www.pscube.jp/dedamajyoho-P-townDMMpachi/c734011
[WAIT] ページ描画待機: 30秒
[BROWSER] browser.quit() 開始
[BROWSER] browser.quit() 完了
[TIME] 実行時間: 4.99秒


KeyboardInterrupt: 